<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: La Línea de Atención de una Cooperativa Ficticia ☎️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 02
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/02_Probabilidad_Distribuciones_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 02: Probabilidad y Distribuciones** ([README del módulo](../02%20-%20Probabilidad%20y%20Distribuciones/README.md)) con un problema **nuevo**: la **línea de atención de una cooperativa ficticia** (alertas de fraude en los pagos, incidentes diarios, llamadas que entran y tiempos de atención).

Al terminar serás capaz de:

1. **Aplicar Bayes e independencia condicional**: construir una tabla conjunta, ver que dos detectores independientes *dado el estado* **no** son independientes entre sí, combinar evidencia y verificarlo por simulación; usar la binomial para reglas de votación.
2. **Trabajar con variables aleatorias**: esperanza y varianza de una variable y de su costo, **Markov y Chebyshev** frente a la probabilidad exacta, covarianza, correlación y esperanza total con una distribución conjunta, y **generar** una variable discreta por transformada inversa.
3. **Elegir y diagnosticar una distribución**: Poisson frente a sobredispersión con una prueba de bondad de ajuste, el exponencial de los tiempos entre llegadas y su conexión con Poisson, y qué aproximación (Poisson o normal) conviene para una binomial.
4. **Aplicar el muestreo y el TLC**: error estándar y asimetría de la media muestral al crecer $n$, probabilidad de superar un umbral, **cobertura real** de un intervalo $t$ con datos asimétricos y tamaño de muestra verificado por simulación.

> 📚 **Repasa si lo necesitas:** [Probabilidad, Bayes e independencia](../02%20-%20Probabilidad%20y%20Distribuciones/00_Probabilidad_Basica_Condicional_Bayes_e_Independencia.ipynb) · [Variables aleatorias](../02%20-%20Probabilidad%20y%20Distribuciones/01_Variables_Aleatorias_Esperanza_y_Varianza.ipynb) · [Distribuciones](../02%20-%20Probabilidad%20y%20Distribuciones/02_Distribuciones_Discretas_y_Continuas.ipynb) · [Muestreo, LGN y TLC](../02%20-%20Probabilidad%20y%20Distribuciones/03_Muestreo_Ley_de_los_Grandes_Numeros_y_TLC.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; la cooperativa y sus cifras son **inventadas**). Bibliotecas: `numpy`, `pandas` y `scipy`. Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math

import numpy as np
import pandas as pd
from scipy import stats

SEMILLA = 2025


def generar_llamadas(semilla=SEMILLA):
    """Llamadas por intervalo de 10 minutos: una serie Poisson, una sobredispersida, y 500 tiempos entre llamadas (min)."""
    rng = np.random.default_rng(semilla)
    ser_a = rng.poisson(4.2, 300)
    ser_b = rng.negative_binomial(3.675, 0.4667, 300)       # misma media (4.2) pero varianza ≈ 9
    gaps = rng.exponential(1 / 0.42, 500)                    # tasa 0.42 llamadas por minuto
    return ser_a, ser_b, gaps


def generar_tiempos(semilla=SEMILLA, N=50_000):
    """Población ficticia de tiempos de atención (minutos): asimétrica a la derecha."""
    return np.random.default_rng(semilla + 5).gamma(1.2, 6.0, N)


print("Listo: generar_llamadas() y generar_tiempos() disponibles (semilla =", SEMILLA, ")")

---
## Reto 1: Dos detectores de fraude que no son independientes 🚨

**Contexto:** la cooperativa revisa cada pago con dos detectores, A y B. El 0.4 % de los pagos es fraude (`PREV`). Si un pago es fraude, A lo marca con probabilidad `SA = 0.90` y B con `SB = 0.80`; si es legítimo, A lo marca por error con probabilidad `FA = 0.05` y B con `FB = 0.02`. Dado el estado del pago (fraude o legítimo), los detectores actúan de forma **independiente**.

**Enunciado**
1. `tabla`: arreglo de forma `(2, 2, 2)` con las probabilidades conjuntas `P[f, a, b]` ($f$ = 1 si hay fraude; $a$, $b$ = 1 si el detector marca). Debe sumar 1.
2. A partir de `tabla`: `p_a` $=P(A)$, `p_b` $=P(B)$, `p_ab` $=P(A\cap B)$ y `cov_ab` $=P(A\cap B)-P(A)P(B)$. Concluye si A y B son independientes **sin condicionar**.
3. `vpp_a` $=P(F\mid A)$, `vpp_ab` $=P(F\mid A\cap B)$ y `vpp_a_noB` $=P(F\mid A\cap \bar B)$ (con la fórmula de Bayes sobre `tabla`).
4. **Simulación:** genera `NSIM = 1_000_000` pagos, marca con A y B, y guarda `n_ab` (cuántos fueron marcados por ambos) y `vpp_ab_sim` (fracción de fraudes entre ellos).
5. **Votación:** `p_falsa_alarma` y `p_deteccion`: probabilidad de que **al menos 2 de 5** detectores independientes disparen, para un pago legítimo (cada detector dispara por error con prob. `FA`) y para un fraude (cada uno acierta con prob. `SA`). Usa `math.comb`.

**Criterios de aceptación:** `tabla` suma 1; `cov_ab > 0` (los detectores se correlacionan a través del fraude); `vpp_ab` mucho mayor que `vpp_a`; la simulación cae dentro de 5 errores estándar de `vpp_ab`; y la votación coincide con `stats.binom.sf(1, 5, p)`.

**Puntaje sugerido:** 20 puntos.

In [ ]:
PREV, SA, SB, FA, FB = 0.004, 0.90, 0.80, 0.05, 0.02
NSIM = 1_000_000

# TODO: completa
tabla = None                    # forma (2, 2, 2): P[f, a, b]
p_a = p_b = p_ab = cov_ab = None
vpp_a = vpp_ab = vpp_a_noB = None

rng = np.random.default_rng(SEMILLA + 1)
n_ab = None                     # pagos marcados por A y por B (simulación)
vpp_ab_sim = None               # fracción de fraudes entre ellos

p_falsa_alarma = p_deteccion = None   # al menos 2 de 5 detectores disparan

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert tabla is not None and tabla.shape == (2, 2, 2) and abs(tabla.sum() - 1) < 1e-12, "tabla debe tener forma (2, 2, 2) y sumar 1: P[f, a, b] = P(f) · P(a | f) · P(b | f) (independencia CONDICIONAL al estado)."
assert abs(tabla[1, 1, 1] - PREV * SA * SB) < 1e-12 and abs(tabla[0, 1, 0] - (1 - PREV) * FA * (1 - FB)) < 1e-12, "Revisa las celdas: P[1,1,1] = PREV·SA·SB y P[0,1,0] = (1-PREV)·FA·(1-FB)."
_pa, _pb = PREV * SA + (1 - PREV) * FA, PREV * SB + (1 - PREV) * FB
_pab = PREV * SA * SB + (1 - PREV) * FA * FB
assert abs(p_a - _pa) < 1e-12 and abs(p_b - _pb) < 1e-12 and abs(p_ab - _pab) < 1e-12, "p_a, p_b y p_ab son sumas de celdas de `tabla` (ley de probabilidad total)."
assert abs(cov_ab - (_pab - _pa * _pb)) < 1e-12 and cov_ab > 0, "cov_ab = P(A∩B) - P(A)P(B) debe ser positiva: A y B NO son independientes si no condicionas al fraude."
assert abs(vpp_a - PREV * SA / _pa) < 1e-12, "vpp_a = P(F|A) = P(F∩A) / P(A) = PREV·SA / P(A) (Bayes)."
assert abs(vpp_ab - PREV * SA * SB / _pab) < 1e-12 and abs(vpp_a_noB - PREV * SA * (1 - SB) / (_pa - _pab)) < 1e-12, "vpp_ab = P(F∩A∩B)/P(A∩B) y vpp_a_noB = P(F∩A∩no B)/P(A∩no B)."
assert vpp_ab > 5 * vpp_a > vpp_a_noB, "Con dos marcas el fraude es mucho más probable; con A y sin B, menos que con A solo."
_se = math.sqrt(vpp_ab * (1 - vpp_ab) / max(n_ab, 1))
assert n_ab is not None and abs(n_ab / NSIM - _pab) < 5 * math.sqrt(_pab * (1 - _pab) / NSIM), "n_ab: cuenta los pagos con a y b verdaderos a la vez (a & b)."
assert abs(vpp_ab_sim - vpp_ab) < 5 * _se, "vpp_ab_sim = fraudes entre los pagos marcados por A y B; debe quedar cerca de vpp_ab (¿usaste fraude[a & b].mean()?)."
_f = lambda p: stats.binom.sf(1, 5, p)
assert abs(p_falsa_alarma - _f(FA)) < 1e-12 and abs(p_deteccion - _f(SA)) < 1e-12, "P(al menos 2 de 5) = suma de comb(5,k) p^k (1-p)^(5-k) para k = 2..5 (o 1 - P(0) - P(1))."
print(f"✅ Reto 1 superado: P(F|A) = {vpp_a:.3f}, P(F|A y B) = {vpp_ab:.3f} (simulado {vpp_ab_sim:.3f} con {n_ab} pagos) | cov(A,B) = {cov_ab:.2e} | votación 2 de 5: falsa alarma {p_falsa_alarma:.4f}, detección {p_deteccion:.5f}")

---
## Reto 2: Incidentes, costos y una distribución conjunta 💰

**Contexto:** sea $X$ el número de **incidentes graves** en un día, con función de masa `pmf` sobre `valores` $=0,\dots,6$. El costo diario de atención es $C=120+80X+300\cdot\mathbf 1\{X\ge 4\}$ (se contrata un refuerzo cuando hay 4 o más). Además, `conjunta` es la distribución conjunta de $X'\in\{0,1,2\}$ (incidentes graves en la mañana) y $W\in\{0,1,2\}$ (escalamientos a gerencia; filas: $X'$, columnas: $W$).

**Enunciado**
1. `esp_x`, `var_x`, `esp_c`, `var_c` (esperanza y varianza de $X$ y de $C$ con la `pmf`) y `esp_c_lineal` $=120+80\,E[X]$ (la esperanza **sin** el refuerzo). Verifica que el refuerzo tiene un costo esperado `costo_refuerzo = esp_c - esp_c_lineal`.
2. `markov`: cota de Markov para $P(X\ge 4)$, y `p_exacta_4` el valor exacto. `cheb`: cota de Chebyshev $1/k^2$ con $k=2$ para $P(|X-\mu|\ge 2\sigma)$, y `p_exacta_cheb` el valor exacto.
3. De `conjunta`: `cov` $=E[X'W]-E[X']E[W]$, `rho` (correlación), `indep` (`True` solo si `conjunta` es igual al producto de sus marginales) y `e_w_dado_x`: arreglo con $E[W\mid X'=x]$ para $x=0,1,2$. Comprueba la **esperanza total**: `suma_total` $=\sum_x P(X'=x)\,E[W\mid X'=x]$.
4. `generar(pmf, n, rng)`: muestras de $X$ por **transformada inversa discreta** (`np.cumsum` y `np.searchsorted` sobre uniformes). Guarda `muestra` ($n=200\,000$) y `frecuencias` (frecuencia relativa de cada valor).

**Criterios de aceptación:** los momentos coinciden con un cálculo independiente; las cotas se cumplen pero son **holgadas**; las dos esperanzas totales coinciden; y las frecuencias simuladas difieren de la `pmf` en menos de 0.01.

**Puntaje sugerido:** 20 puntos.

In [ ]:
valores = np.arange(7)
pmf = np.array([0.30, 0.28, 0.20, 0.12, 0.06, 0.03, 0.01])
costo = 120 + 80 * valores + 300 * (valores >= 4)
conjunta = np.array([[0.30, 0.10, 0.02],
                     [0.08, 0.20, 0.07],
                     [0.01, 0.07, 0.15]])

# TODO: completa
esp_x = var_x = esp_c = var_c = esp_c_lineal = costo_refuerzo = None
markov = p_exacta_4 = cheb = p_exacta_cheb = None
cov = rho = indep = None
e_w_dado_x = suma_total = None


def generar(pmf, n, rng):
    # TODO: transformada inversa: uniformes + searchsorted sobre la acumulada
    return None


rng = np.random.default_rng(SEMILLA + 2)
muestra = None             # 200 000 valores
frecuencias = None         # frecuencia relativa de cada valor 0..6

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_m = (valores * pmf).sum(); _v = ((valores - _m) ** 2 * pmf).sum()
assert esp_x is not None and abs(esp_x - _m) < 1e-12 and abs(var_x - _v) < 1e-12, "esp_x = Σ x·p(x) y var_x = Σ (x - esp_x)²·p(x) (o E[X²] - E[X]²)."
_c = (costo * pmf).sum(); _vc = ((costo - _c) ** 2 * pmf).sum()
assert abs(esp_c - _c) < 1e-12 and abs(var_c - _vc) < 1e-12, "esp_c y var_c: lo mismo con el vector `costo` (que ya incluye el refuerzo) y la misma pmf."
assert abs(esp_c_lineal - (120 + 80 * _m)) < 1e-12 and abs(costo_refuerzo - (_c - 120 - 80 * _m)) < 1e-12 and costo_refuerzo > 0, "esp_c_lineal = 120 + 80·E[X] (linealidad) y costo_refuerzo = esp_c - esp_c_lineal = 300·P(X >= 4)."
assert abs(costo_refuerzo - 300 * pmf[4:].sum()) < 1e-9, "El costo esperado del refuerzo debe ser 300 · P(X >= 4)."
assert abs(markov - _m / 4) < 1e-12 and abs(p_exacta_4 - pmf[4:].sum()) < 1e-12 and p_exacta_4 <= markov, "Markov: P(X >= 4) <= E[X] / 4; la cota debe cumplirse."
_pc = pmf[np.abs(valores - _m) >= 2 * math.sqrt(_v)].sum()
assert abs(cheb - 0.25) < 1e-12 and abs(p_exacta_cheb - _pc) < 1e-12 and p_exacta_cheb <= cheb, "Chebyshev: P(|X - μ| >= 2σ) <= 1/4; el valor exacto suma la pmf de los x con |x - μ| >= 2σ."
assert p_exacta_4 < markov / 2 and p_exacta_cheb < cheb, "Las cotas son holgadas: Markov debería exceder por mucho a la probabilidad exacta."
_px, _pw = conjunta.sum(1), conjunta.sum(0); _x = np.arange(3)
_cov = (np.outer(_x, _x) * conjunta).sum() - (_x * _px).sum() * (_x * _pw).sum()
assert cov is not None and abs(cov - _cov) < 1e-12, "cov = E[X'W] - E[X']E[W], con E[X'W] = Σ x·w·p(x, w) (np.outer)."
_rho = _cov / math.sqrt((((_x - (_x * _px).sum()) ** 2) * _px).sum() * (((_x - (_x * _pw).sum()) ** 2) * _pw).sum())
assert abs(rho - _rho) < 1e-12 and 0 < rho < 1, "rho = cov / (σ_X' · σ_W), entre 0 y 1 aquí."
assert indep is False, "indep debe ser False: conjunta != producto de marginales (np.allclose(conjunta, np.outer(px, pw)))."
assert np.allclose(e_w_dado_x, (conjunta * _x).sum(1) / _px), "E[W | X'=x] = Σ_w w·p(x, w) / P(X'=x)."
assert abs(suma_total - (_x * _pw).sum()) < 1e-12, "Esperanza total: Σ_x P(X'=x)·E[W|X'=x] debe igualar E[W]."
assert muestra is not None and len(muestra) == 200_000 and set(np.unique(muestra)) <= set(range(7)), "generar debe devolver enteros 0..6: np.searchsorted(np.cumsum(pmf), uniformes)."
assert frecuencias is not None and np.max(np.abs(frecuencias - pmf)) < 0.01 and abs(frecuencias.sum() - 1) < 1e-12, "Las frecuencias relativas de la muestra deben parecerse a la pmf (diferencia máxima < 0.01); usa np.bincount(muestra, minlength=7) / n."
print(f"✅ Reto 2 superado: E[X] = {esp_x:.2f}, Var[X] = {var_x:.2f} | E[C] = {esp_c:.1f} (refuerzo aporta {costo_refuerzo:.1f}) | Markov {markov:.3f} vs {p_exacta_4:.3f}, Chebyshev {cheb:.2f} vs {p_exacta_cheb:.3f} | cov = {cov:.3f}, rho = {rho:.2f}")

---
## Reto 3: Llamadas, tiempos entre llamadas y aproximaciones 📞

**Contexto:** `ser_a` y `ser_b` cuentan las llamadas recibidas en 300 intervalos de 10 minutos en dos líneas distintas; `gaps` son 500 tiempos (min) entre llamadas consecutivas de la línea A. Una línea sigue un proceso de Poisson y la otra tiene **sobredispersión**. También quieres aproximar la probabilidad de que, entre 500 pagos con 2 % de marcas, haya **12 o más** marcados.

**Enunciado**
1. `lam_a`, `lam_b` (medias) y los **índices de dispersión** `disp_a`, `disp_b` $=s^2/\bar x$ (varianza con `ddof=1`).
2. **Bondad de ajuste Poisson** con las 10 clases $0,1,\dots,8$ y $\ge 9$: `chi2_gof(serie)` $\to(\chi^2, \text{gl}, p)$ con esperados $n\cdot P(X=k)$ de una Poisson de media `serie.mean()`, y `gl` $=$ clases $-1-1$ (un parámetro estimado). Guarda `res_a` y `res_b`.
3. Sobre `gaps`: `escala` (media), `ks_stat` (estadístico de Kolmogorov-Smirnov **desde cero** contra la exponencial de esa escala: $D=\max_i\max\{i/n-F(x_{(i)}),\,F(x_{(i)})-(i-1)/n\}$), `p0_poisson` $=e^{-\lambda_A}$ y `p0_exp` $=e^{-10/\text{escala}}$ (probabilidad de **cero llamadas** en 10 min por dos caminos).
4. `p_exacta` $=P(X\ge12)$ con $X\sim\text{Bin}(500,0.02)$, `p_poisson` (Poisson de media $np$) y `p_normal` (normal con corrección por continuidad: $P(X\ge12)\approx P(Z\ge (11.5-np)/\sqrt{npq})$); `mejor`: `'poisson'` o `'normal'`, la aproximación más cercana a `p_exacta`.

**Criterios de aceptación:** los índices y la prueba coinciden con un cálculo independiente; la línea A no se rechaza como Poisson y la B sí; `ks_stat` coincide con `stats.kstest`; y las dos rutas a «cero llamadas» concuerdan.

**Puntaje sugerido:** 20 puntos.

In [ ]:
ser_a, ser_b, gaps = generar_llamadas()

# TODO: completa
lam_a = lam_b = disp_a = disp_b = None


def chi2_gof(serie):
    # TODO: clases 0..8 y >=9; esperados n·pmf de Poisson(media de la serie); devuelve (chi2, gl, p)
    return None, None, None


res_a = res_b = None

escala = ks_stat = None
p0_poisson = p0_exp = None

p_exacta = p_poisson = p_normal = None
mejor = None       # 'poisson' o 'normal'

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert lam_a is not None and abs(lam_a - ser_a.mean()) < 1e-12 and abs(disp_a - ser_a.var(ddof=1) / ser_a.mean()) < 1e-12 and abs(disp_b - ser_b.var(ddof=1) / ser_b.mean()) < 1e-12, "disp = varianza muestral (ddof=1) / media: ≈ 1 para una Poisson."
assert 0.7 < disp_a < 1.4 and disp_b > 1.5, "La línea A debe tener índice de dispersión cercano a 1 y la B claramente mayor que 1 (sobredispersión)."
def _gof(s):
    n, l = len(s), s.mean(); o = np.bincount(np.minimum(s, 9), minlength=10)
    e = n * np.append(stats.poisson.pmf(np.arange(9), l), stats.poisson.sf(8, l))
    c = ((o - e) ** 2 / e).sum(); return c, 8, stats.chi2.sf(c, 8)
for _nom, _r, _s in (("res_a", res_a, ser_a), ("res_b", res_b, ser_b)):
    _e = _gof(_s)
    assert _r is not None and abs(_r[0] - _e[0]) < 1e-8 and _r[1] == 8 and abs(_r[2] - _e[2]) < 1e-10, f"{_nom} debe ser (chi2, 8, p): 10 clases (0..8 y >=9; usa np.minimum(serie, 9)), esperados n·P(X=k) y la cola ≥ 9 con poisson.sf(8, media); gl = 10 - 1 - 1."
assert res_a[2] > 0.001 and res_b[2] < 0.001, "La línea A (Poisson verdadera) no debería rechazarse y la B (sobredispersa) sí."
assert abs(escala - gaps.mean()) < 1e-12, "escala = media de gaps (estimador de máxima verosimilitud de la escala exponencial)."
assert abs(ks_stat - stats.kstest(gaps, "expon", args=(0, escala)).statistic) < 1e-9, "ks_stat: ordena gaps, F = 1 - exp(-x/escala) y toma el mayor entre max(i/n - F) y max(F - (i-1)/n)."
assert abs(p0_poisson - math.exp(-ser_a.mean())) < 1e-12 and abs(p0_exp - math.exp(-10 / gaps.mean())) < 1e-12, "p0_poisson = exp(-lam_a) y p0_exp = exp(-10/escala)."
assert abs(p0_poisson - p0_exp) < 0.01, "Poisson para conteos y exponencial para tiempos entre llegadas describen el mismo proceso: P(0 llamadas en 10 min) debe coincidir por los dos caminos."
assert abs(p_exacta - stats.binom.sf(11, 500, 0.02)) < 1e-12, "p_exacta = P(X >= 12) = binom.sf(11, 500, 0.02)  (sf(k) = P(X > k))."
assert abs(p_poisson - stats.poisson.sf(11, 10)) < 1e-12, "p_poisson = poisson.sf(11, 500·0.02)."
assert abs(p_normal - stats.norm.sf((11.5 - 10) / math.sqrt(500 * 0.02 * 0.98))) < 1e-12, "p_normal = norm.sf((11.5 - n·p) / sqrt(n·p·q)) (11.5 es la corrección por continuidad para P(X >= 12))."
assert mejor == "poisson", "Con p pequeño (0.02) y n grande la binomial se parece más a una Poisson que a una normal: la aproximación normal sufre por la asimetría."
print(f"✅ Reto 3 superado: dispersión A {disp_a:.2f} (p = {res_a[2]:.3f}) frente a B {disp_b:.2f} (p = {res_b[2]:.2g}) | KS = {ks_stat:.3f} | P(0 llamadas) {p0_poisson:.4f} (Poisson) = {p0_exp:.4f} (exp) | P(X>=12): exacta {p_exacta:.4f}, Poisson {p_poisson:.4f}, normal {p_normal:.4f}")

---
## Reto 4: Promedios de tiempos asimétricos: error, TLC y cobertura 🔔

**Contexto:** `tiempos` es la población de 50 000 tiempos de atención (minutos), muy asimétrica a la derecha. Vas a estudiar cómo se comporta la **media muestral** al crecer $n$, a usar el **TLC** para estimar una probabilidad, a medir la **cobertura real** de un intervalo $t$ del 95 % y a verificar un **tamaño de muestra**. Muestrea siempre **con reemplazo** (`rng.choice(tiempos, size=(reps, n))`) y trabaja por filas.

**Enunciado**
1. `mu`, `sigma` (desviación poblacional, `ddof=0`) y `g1_pob` (asimetría de la población, `stats.skew`).
2. Para `n` en `(5, 30, 200)` y `REPS = 4000` muestras: diccionarios `se_emp[n]` (desviación estándar de las medias), `se_teo[n]` $=\sigma/\sqrt n$ y `asim_medias[n]` (asimetría de las medias; la teoría dice $\gamma_1/\sqrt n$).
3. `p_tlc`: aproximación normal de $P(\bar X_{40}>8.5)$; `p_sim`: la fracción de 20 000 medias de muestras de tamaño 40 que superan 8.5.
4. `cobertura[n]` para `n` en `(10, 100)`: fracción de 4000 intervalos $\bar x\pm t_{0.975,\,n-1}\,s/\sqrt n$ (con $s$ de `ddof=1`) que contienen a `mu`.
5. `n_req` $=\lceil (1.96\,\sigma/E)^2\rceil$ para `E = 0.5` minutos y `prop_ok`: fracción de 3000 muestras de tamaño `n_req` cuya media queda a menos de `E` de `mu`.

**Criterios de aceptación:** `se_emp` queda a menos de 6 % de `se_teo`; la asimetría de las medias se parece a $\gamma_1/\sqrt n$ y decrece; `p_tlc` y `p_sim` difieren en menos de 0.02; con $n=10$ la cobertura del intervalo $t$ es **menor que 0.95** y mejora con $n$; y `prop_ok` cae entre 0.93 y 0.97.

**Puntaje sugerido:** 25 puntos.

In [ ]:
tiempos = generar_tiempos()
rng = np.random.default_rng(SEMILLA + 3)
REPS = 4000
E = 0.5

# TODO: completa
mu = sigma = g1_pob = None
se_emp, se_teo, asim_medias = {}, {}, {}     # claves: n = 5, 30, 200
for n in (5, 30, 200):
    pass

p_tlc = p_sim = None                          # P(media de n=40 > 8.5)

cobertura = {}                                # claves: n = 10, 100
n_req = prop_ok = None

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert mu is not None and abs(mu - tiempos.mean()) < 1e-9 and abs(sigma - tiempos.std()) < 1e-9 and abs(g1_pob - stats.skew(tiempos)) < 1e-9, "mu = media, sigma = desviación con ddof=0 (es la población completa) y g1_pob = stats.skew."
for _n in (5, 30, 200):
    assert _n in se_emp and abs(se_teo[_n] - sigma / math.sqrt(_n)) < 1e-12, f"se_teo[{_n}] = sigma / sqrt({_n})."
    assert abs(se_emp[_n] / se_teo[_n] - 1) < 0.06, f"El error estándar simulado con n = {_n} debe quedar a menos de 6 % de sigma/sqrt(n): ¿tomaste la media de cada fila (axis=1) y muestreaste con reemplazo?"
    assert abs(asim_medias[_n] - g1_pob / math.sqrt(_n)) < 0.15, f"La asimetría de las medias con n = {_n} debe parecerse a g1_pob / sqrt(n) = {g1_pob / math.sqrt(_n):.2f}."
assert asim_medias[5] > asim_medias[30] > asim_medias[200], "La asimetría de la distribución de la media debe decrecer al aumentar n (el TLC en acción)."
assert abs(p_tlc - stats.norm.sf((8.5 - tiempos.mean()) / (tiempos.std() / math.sqrt(40)))) < 1e-9, "p_tlc = norm.sf((8.5 - mu) / (sigma / sqrt(40)))."
assert p_sim is not None and abs(p_tlc - p_sim) < 0.02, "p_sim = fracción de 20 000 medias (muestras de 40) mayores que 8.5; debe estar cerca de p_tlc."
assert set(cobertura) == {10, 100} and cobertura[10] < 0.95 and cobertura[100] < 0.965 and 0.86 < cobertura[10] < 0.935 and cobertura[10] < cobertura[100], "El intervalo t con datos asimétricos cubre MENOS del 95 % con n = 10 y mejora con n = 100: revisa t.ppf(0.975, n-1) y que s use ddof=1."
assert n_req == math.ceil((1.96 * tiempos.std() / 0.5) ** 2), "n_req = techo((1.96 · sigma / E)²)."
assert prop_ok is not None and 0.93 < prop_ok < 0.97, "Con ese tamaño, ≈ 95 % de las medias deben quedar a menos de E de mu."
print(f"✅ Reto 4 superado: g1 población {g1_pob:.2f} | asimetría de las medias {asim_medias[5]:.2f}→{asim_medias[30]:.2f}→{asim_medias[200]:.2f} | P(media>8.5): TLC {p_tlc:.3f}, simulación {p_sim:.3f} | cobertura t: n=10 {cobertura[10]:.3f}, n=100 {cobertura[100]:.3f} | n_req = {n_req} ({prop_ok:.3f} dentro de ±{E})")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Por qué un solo detector con $P(F\mid A)=$ `vpp_a` no basta y por qué `vpp_ab` sube tanto? ¿Por qué la covarianza de A y B es positiva si «actúan independientemente»? ¿Qué supuesto sería falso si los dos detectores usaran la misma regla?
- Con tus cifras, ¿cuánto del costo esperado diario viene del refuerzo? ¿Qué te dicen Markov y Chebyshev frente a las probabilidades exactas, y cuándo servirían de todos modos?
- ¿Qué diagnóstico distingue a la línea A de la B (índice de dispersión y prueba chi-cuadrado) y qué consecuencias prácticas tiene la sobredispersión para planear la dotación de personal? ¿Por qué `p0_poisson` y `p0_exp` coinciden?
- ¿Por qué la aproximación de Poisson le ganó a la normal en el último paso y cuándo ocurriría lo contrario?
- Con tus resultados de `cobertura` y `asim_medias`, ¿cuándo puedes confiar en el TLC para este tipo de datos y cómo cambia con $n$? ¿Qué verificó `prop_ok`?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: tabla conjunta, Bayes con dos detectores, simulación y votación binomial | 20 |
| Reto 2: momentos de una variable y de su costo, Markov/Chebyshev, conjunta, esperanza total e inversa | 20 |
| Reto 3: dispersión, bondad de ajuste, exponencial/Poisson y aproximaciones de la binomial | 20 |
| Reto 4: error estándar, asimetría de las medias, TLC, cobertura y tamaño de muestra | 25 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Comprobé que mis probabilidades suman 1 y que las cotas (Markov, Chebyshev) se cumplen.
- [ ] Distingo en mis palabras la independencia condicional de la independencia marginal.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>